In [2]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ==============================================================================
# 1. LOAD DATASET AND STANDARDISE HEADERS
# ==============================================================================
# Update the filename to match your local CSV file
df7 = pd.read_csv("Detailed_Polling_Data.csv")

# Standardise column spacing and strip whitespace to prevent key errors
df7.columns = df7.columns.str.replace(r'\s+', ' ', regex=True).str.strip()

# Clear out any pre-existing calculated column copies to prevent structural duplication errors
cols_to_clear = ['Margin_Percentage', 'Winner_Votes', 'Runner_Up_Votes', 'Margin_Of_Victory', 'Winner_Party', 'Cluster_ID']
df7 = df7.drop(columns=[c for c in cols_to_clear if c in df7.columns], errors='ignore')

# ==============================================================================
# 2. DEFINE DATASET CORE PARAMETERS
# ==============================================================================
core_parties = [
    'Dravida Munnetra Kazhagam',
    'All India Anna Dravida Munnetra Kazhagam', 
    'Tamilaga Vettri Kazhagam',
    'Naam Tamilar Katchi', 
    'Bahujan Samaj Party', 
    'Tamizhaga Vaazhvurimai Katchi',
    'Samaniya Makkal Nala Katchi'
]
df7[core_parties] = df7[core_parties].fillna(0)

# Exact structural column definitions from your current dataset
station_col = 'Serial No. Of Polling Station'
building_col = 'Location and Name of Building in which Polling Station Located'
area_col = 'Polling Area'

# Isolate the 6 explicit independent candidate fields present in this specific layout
independent_candidate_cols = ['Independent', 'Independent.1', 'Independent.2', 'Independent.3', 'Independent.4', 'Independent.5']
existing_ind_cols = [c for c in independent_candidate_cols if c in df7.columns]
df7[existing_ind_cols] = df7[existing_ind_cols].fillna(0)

# Calculate total independent votes dynamically
df7['Total_Independent_Votes'] = df7[existing_ind_cols].sum(axis=1)

# ==============================================================================
# 3. CALCULATE NORMALIZED METRICS FOR THE MACHINE LEARNING MODEL
# ==============================================================================
# Calculate true total votes for normalization (Core Parties + Independents + NOTA)
df7['Total_Calculated_Votes'] = df7[core_parties].sum(axis=1) + df7['Total_Independent_Votes'] + df7['NOTA'].fillna(0)

# Filter out empty entries to completely avoid division by zero errors
df7 = df7[df7['Total_Calculated_Votes'] > 0].copy()

# Explicit Abbreviation Mapping to keep feature streams completely unique
party_abbreviations = {
    'Dravida Munnetra Kazhagam': 'DMK',
    'All India Anna Dravida Munnetra Kazhagam': 'AIADMK',
    'Tamilaga Vettri Kazhagam': 'TVK',
    'Naam Tamilar Katchi': 'NTK',
    'Bahujan Samaj Party': 'BSP',
    'Tamizhaga Vaazhvurimai Katchi': 'TAVAK',
    'Samaniya Makkal Nala Katchi': 'SMNK'
}

# Feature Engineering: Create normalized percentage shares (%)
share_cols = []
for party in core_parties:
    party_label = party_abbreviations[party]
    col_name = f'{party_label}_share_pct'
    
    if col_name in df7.columns:
        df7 = df7.drop(columns=[col_name])
        
    df7[col_name] = (df7[party] / df7['Total_Calculated_Votes']) * 100
    share_cols.append(col_name)

df7['independent_share_pct'] = (df7['Total_Independent_Votes'] / df7['Total_Calculated_Votes']) * 100

# Primary voting calculations (Winner, Runner-up, Margin)
df7['Winner_Votes'] = df7[core_parties].max(axis=1)
sorted_votes = np.sort(df7[core_parties].values, axis=1)
df7['Runner_Up_Votes'] = sorted_votes[:, -2]
df7['Margin_Of_Victory'] = df7['Winner_Votes'] - df7['Runner_Up_Votes']
df7['Winner_Party'] = df7[core_parties].idxmax(axis=1)
df7['Margin_Percentage'] = (df7['Margin_Of_Victory'] / df7['Total_Calculated_Votes']) * 100

# Set up clean target feature tracking array
feature_cols = share_cols + ['independent_share_pct', 'Margin_Percentage']
X = df7[feature_cols].copy().fillna(0)

# ==============================================================================
# 4. SCALE FEATURES AND RUN K-MEANS CLUSTERING
# ==============================================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
df7['Cluster_ID'] = kmeans.fit_predict(X_scaled)

# ==============================================================================
# 5. PRINT THE PROFILE SUMMARY BREAKDOWNS
# ==============================================================================
print("\n--- DATASET 7: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---")
print(df7.groupby('Cluster_ID')[feature_cols].mean().round(2))

print("\n--- DATASET 7: BOOTH COUNT PER CLUSTER ---")
print(df7['Cluster_ID'].value_counts())

# ==============================================================================
# 6. EXPORT STRATEGIC TARGET SHEETS FOR GROUND CAMPAIGN TEAMS
# ==============================================================================
# Note: You can rename these targets in this dictionary based on your metrics later
cluster_names = {
    0: "Cluster_0_Target", 
    1: "Cluster_1_Target", 
    2: "Cluster_2_Target", 
    3: "Cluster_3_Target"
}

for cluster_num in range(optimal_k):
    target_cols = [station_col, building_col, area_col, 'Winner_Party', 'Margin_Percentage']
    valid_target_cols = [c for c in target_cols if c in df7.columns]
    
    cluster_df = df7[df7['Cluster_ID'] == cluster_num][valid_target_cols]
    filename = f"Dataset_7_Cluster_{cluster_num}_{cluster_names[cluster_num]}.csv"
    cluster_df.to_csv(filename, index=False)

print("\nSuccess! Campaign target files generated cleanly for all 4 clusters.")



--- DATASET 7: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---
            DMK_share_pct  AIADMK_share_pct  TVK_share_pct  NTK_share_pct  \
Cluster_ID                                                                  
0                   29.61             34.32          30.26           2.60   
1                   27.61             30.95          35.48           4.20   
2                   37.08             29.51          29.23           2.78   
3                   26.22             45.46          23.57           3.10   

            BSP_share_pct  TAVAK_share_pct  SMNK_share_pct  \
Cluster_ID                                                   
0                    0.57             0.13            0.06   
1                    0.21             0.22            0.03   
2                    0.16             0.08            0.03   
3                    0.29             0.10            0.04   

            independent_share_pct  Margin_Percentage  
Cluster_ID                                      